In [1]:
import os
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
CLASS_MAP = {
    "N": 0,
    "A": 1,
    "V": 2,
    "F": 3,
    "f": 4,
    "R": 5,
    "B": 6,
    "L": 7
}

In [3]:
def list_csv_files(root):
    csv_files = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(".csv"):
                csv_files.append(os.path.join(dirpath, f))
    return csv_files



In [4]:
class ECGDataset(Dataset):
    def __init__(self, files, class_map, fixed_len=600):
        self.files = files
        self.class_map = class_map
        self.fixed_len = fixed_len

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]

        df = pd.read_csv(path)

        # usa a primeira coluna (seu CSV real)
        signal = df.iloc[:, 0].values.astype(np.float32)

        signal = pad_or_crop(signal, self.fixed_len)

        x = torch.tensor(signal).unsqueeze(0)  # (1, L)

        label_char = os.path.basename(path)[0]
        y = torch.tensor(self.class_map[label_char], dtype=torch.long)

        return x, y


In [5]:
class ECGCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.classifier(x)


In [6]:
def pad_or_crop(signal, target_len):
    length = len(signal)

    if length == target_len:
        return signal

    if length > target_len:
        start = (length - target_len) // 2
        return signal[start:start + target_len]

    # padding
    pad_width = target_len - length
    left = pad_width // 2
    right = pad_width - left
    return np.pad(signal, (left, right), mode="constant")


In [7]:
def train_model(model, train_loader, val_loader, device, epochs=30):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                preds = output = model(x).argmax(1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss: {total_loss:.3f} | "
            f"Val Acc: {correct/total:.4f}"
        )


In [9]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_files = list_csv_files("dataset_split_2/train")
    val_files   = list_csv_files("dataset_split_2/val")

    train_dataset = ECGDataset(train_files, CLASS_MAP)
    val_dataset   = ECGDataset(val_files, CLASS_MAP)


    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )


    model = ECGCNN(num_classes=len(CLASS_MAP)).to(device)

    train_model(model, train_loader, val_loader, device)


Device: cuda
Epoch 01 | Train Loss: 809.855 | Val Acc: 0.7101
Epoch 02 | Train Loss: 442.888 | Val Acc: 0.8254
Epoch 03 | Train Loss: 317.276 | Val Acc: 0.7238
Epoch 04 | Train Loss: 262.857 | Val Acc: 0.7849
Epoch 05 | Train Loss: 237.301 | Val Acc: 0.8784
Epoch 06 | Train Loss: 211.343 | Val Acc: 0.8855
Epoch 07 | Train Loss: 199.269 | Val Acc: 0.8785
Epoch 08 | Train Loss: 190.227 | Val Acc: 0.8914
Epoch 09 | Train Loss: 180.290 | Val Acc: 0.8576
Epoch 10 | Train Loss: 176.767 | Val Acc: 0.7773
Epoch 11 | Train Loss: 168.890 | Val Acc: 0.9076
Epoch 12 | Train Loss: 160.325 | Val Acc: 0.8912
Epoch 13 | Train Loss: 159.850 | Val Acc: 0.8999
Epoch 14 | Train Loss: 151.563 | Val Acc: 0.9019
Epoch 15 | Train Loss: 150.148 | Val Acc: 0.8632
Epoch 16 | Train Loss: 145.023 | Val Acc: 0.9002
Epoch 17 | Train Loss: 144.158 | Val Acc: 0.9246
Epoch 18 | Train Loss: 137.952 | Val Acc: 0.9043
Epoch 19 | Train Loss: 136.236 | Val Acc: 0.9236
Epoch 20 | Train Loss: 130.374 | Val Acc: 0.9209
Epoch 2